### スクレイピング

In [34]:
import requests
from bs4 import BeautifulSoup
import json
import time
import math

# サーバー負荷軽減のための待機時間（GitHubの利用規約を尊重）
REQUEST_DELAY_SECONDS = 0.5 

def get_repository_data(page_url, headers):
    """指定されたURLからHTMLを取得し、埋め込みJSONを抽出してリポジトリリストを返します。"""
    
    # 実際のリクエストには待機を入れます
    time.sleep(REQUEST_DELAY_SECONDS)

    try:
        response = requests.get(page_url, headers=headers)
        response.raise_for_status() # HTTPエラーがあれば例外を発生させる
    except requests.exceptions.RequestException as e:
        # エラー発生時はNoneを返して上位関数に処理を委ねる
        return None, None, None

    soup = BeautifulSoup(response.text, 'html.parser')
    json_script_tag = soup.find('script', {'data-target': 'react-app.embeddedData'})
    
    if not json_script_tag or not json_script_tag.string:
        return None, None, None

    try:
        data = json.loads(json_script_tag.string)
        page_data = data.get('payload', {}).get('orgReposPageRoute', {})
        
        # リポジトリのリスト、総リポジトリ数、総ページ数を抽出
        repositories = page_data.get('repositories', [])
        total_repo_count = page_data.get('repositoryCount', 0)
        total_pages = page_data.get('pageCount', 1)

        return repositories, total_repo_count, total_pages

    except (json.JSONDecodeError, KeyError):
        # JSON解析エラーやキーエラーが発生した場合
        return None, None, None


def scrape_and_output_all_google_repositories(base_url):
    """
    GitHubの全リポジトリ情報をスクレイピングし、一件ずつコンソールに出力します。
    """
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
    }
    
    current_page = 1
    total_pages = 1 
    repo_counter = 0

    while current_page <= total_pages:
        target_url = f"{base_url}&page={current_page}"
        
        repo_list, total_count, calculated_pages = get_repository_data(target_url, headers)

        if repo_list is None:
            # ページ取得またはJSON解析でエラーが発生した場合、処理を終了
            print("⚠️ データの取得または解析中にエラーが発生したため、処理を中断しました。")
            break
            
        if current_page == 1 and calculated_pages is not None:
            # 1ページ目でのみ総ページ数を設定
            total_pages = calculated_pages
        
        # 取得したリポジトリ情報を一件ずつ出力
        for repo in repo_list:
            repo_counter += 1
            repo_name = repo.get('name')
            
            # 主要な言語を抽出
            primary_language = repo.get('primaryLanguage')
            language_name = primary_language.get('name') if primary_language else "N/A"
            
            # スター数を抽出し、表示形式を整える (例: 15415 -> 15.4k)
            stars_count = repo.get('starsCount', 0)
            
            if stars_count >= 1000:
                # 1000以上の場合はK表記に丸める
                display_stars = f"{round(stars_count / 1000, 1)}k"
            else:
                display_stars = str(stars_count)
            
            # 一件ずつコンソールに出力
            print(f"[{repo_counter}件目]")
            print(f"  リポジトリ名: {repo_name}")
            print(f"  主要な言語: {language_name}")
            print(f"  スターの数: {display_stars}")
        
        # 次のページへ
        current_page += 1
        
        if not repo_list and current_page > 1:
            break

    print("==================================================")
    if repo_counter > 0:
        print(f"🎉 データ取得と出力が完了しました。総件数: {repo_counter}件")
    elif current_page == 1 and repo_counter == 0:
        print("⚠️ リポジトリが見つかりませんでした。GitHubのHTML構造が再度変更された可能性があります。")


# --- 実行部分 ---
BASE_URL = "https://github.com/orgs/google/repositories?q=&type=all&language=&sort=stargazers"
scrape_and_output_all_google_repositories(BASE_URL)

[1件目]
  リポジトリ名: material-design-icons
  主要な言語: N/A
  スターの数: 52.5k
[2件目]
  リポジトリ名: guava
  主要な言語: Java
  スターの数: 51.3k
[3件目]
  リポジトリ名: zx
  主要な言語: JavaScript
  スターの数: 44.9k
[4件目]
  リポジトリ名: styleguide
  主要な言語: HTML
  スターの数: 38.7k
[5件目]
  リポジトリ名: leveldb
  主要な言語: C++
  スターの数: 38.4k
[6件目]
  リポジトリ名: googletest
  主要な言語: C++
  スターの数: 37.5k
[7件目]
  リポジトリ名: comprehensive-rust
  主要な言語: Rust
  スターの数: 32.3k
[8件目]
  リポジトリ名: material-design-lite
  主要な言語: HTML
  スターの数: 32.2k
[9件目]
  リポジトリ名: python-fire
  主要な言語: Python
  スターの数: 28.0k
[10件目]
  リポジトリ名: flatbuffers
  主要な言語: C++
  スターの数: 25.1k
[11件目]
  リポジトリ名: gson
  主要な言語: Java
  スターの数: 24.2k
[12件目]
  リポジトリ名: ExoPlayer
  主要な言語: Java
  スターの数: 21.9k
[13件目]
  リポジトリ名: iosched
  主要な言語: Kotlin
  スターの数: 21.8k
[14件目]
  リポジトリ名: eng-practices
  主要な言語: N/A
  スターの数: 20.4k
[15件目]
  リポジトリ名: fonts
  主要な言語: HTML
  スターの数: 19.4k
[16件目]
  リポジトリ名: filament
  主要な言語: C++
  スターの数: 19.1k
[17件目]
  リポジトリ名: cadvisor
  主要な言語: Go
  スターの数: 18.6k
[18件目]
  リポジトリ名: web-starter-kit
  主要な言

### DB関連

In [1]:
import requests
from bs4 import BeautifulSoup
import json
import time
import sqlite3 # データベース操作
import math

# サーバー負荷軽減のための待機時間（GitHubの利用規約を尊重）
REQUEST_DELAY_SECONDS = 1.0 # 1秒待機
DB_NAME = 'github_repos.db' # データベースファイル名

# --- 既存のスクレイピングヘルパー関数（変更なし） ---

def get_repository_data(page_url, headers):
    """指定されたURLからHTMLを取得し、埋め込みJSONを抽出してリポジトリリストを返します。"""
    
    # 実際のリクエストには待機を入れます
    time.sleep(REQUEST_DELAY_SECONDS)

    try:
        response = requests.get(page_url, headers=headers)
        response.raise_for_status() # HTTPエラーがあれば例外を発生させる
    except requests.exceptions.RequestException as e:
        # エラー発生時はNoneを返して上位関数に処理を委ねる
        print(f"⚠️ リクエストエラーが発生しました: {e}")
        return None, None, None

    soup = BeautifulSoup(response.text, 'html.parser')
    json_script_tag = soup.find('script', {'data-target': 'react-app.embeddedData'})
    
    if not json_script_tag or not json_script_tag.string:
        return None, None, None

    try:
        data = json.loads(json_script_tag.string)
        page_data = data.get('payload', {}).get('orgReposPageRoute', {})
        
        # リポジトリのリスト、総リポジトリ数、総ページ数を抽出
        repositories = page_data.get('repositories', [])
        total_repo_count = page_data.get('repositoryCount', 0)
        total_pages = page_data.get('pageCount', 1)

        return repositories, total_repo_count, total_pages

    except (json.JSONDecodeError, KeyError) as e:
        # JSON解析エラーやキーエラーが発生した場合
        print(f"⚠️ JSON解析エラーまたはキーエラーが発生しました: {e}")
        return None, None, None

# --- 新しいDB関連関数とメインの実行関数 ---

## データベースのセットアップ
def setup_database():
    """データベースに接続し、テーブルが存在しなければ作成します。"""
    conn = None
    try:
        # 接続
        conn = sqlite3.connect(DB_NAME)
        cur = conn.cursor()
        
        # テーブル作成（存在しない場合のみ実行: IF NOT EXISTS）
        # nameをPRIMARY KEYとし、重複挿入を防ぎます。
        sql = '''
        CREATE TABLE IF NOT EXISTS repositories (
            name TEXT PRIMARY KEY,
            primary_language TEXT,
            stars_count INTEGER
        );
        '''
        cur.execute(sql)
        # 変更を確定
        conn.commit()
        print(f"✅ データベース `{DB_NAME}` に接続し、テーブルの準備が完了しました。")
        return conn
    except sqlite3.Error as e:
        print(f"❌ データベース接続またはテーブル作成時にエラーが発生しました: {e}")
        if conn:
            conn.close()
        return None

## スクレイピングとDB保存を実行するメイン関数
def scrape_and_save_all_google_repositories(base_url):
    """
    GitHubの全リポジトリ情報をスクレイピングし、データベースに保存します。
    """
    # DB接続を試みる
    conn = setup_database()
    if conn is None:
        return # DB接続失敗時は処理を終了

    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
    }
    
    current_page = 1
    total_pages = 1 
    repo_counter = 0
    all_repo_data = [] # 挿入するデータをタプルのリストとして保持

    try:
        while current_page <= total_pages:
            target_url = f"{base_url}&page={current_page}"
            
            # スクレイピング実行
            repo_list, total_count, calculated_pages = get_repository_data(target_url, headers)

            if repo_list is None:
                print("⚠️ データの取得または解析中にエラーが発生したため、データ保存処理を中断します。")
                break
                
            if current_page == 1 and calculated_pages is not None:
                total_pages = calculated_pages
            
            # データを整形し、挿入リストに追加
            for repo in repo_list:
                repo_counter += 1
                repo_name = repo.get('name')
                
                primary_language = repo.get('primaryLanguage')
                language_name = primary_language.get('name') if primary_language else "N/A"
                
                stars_count = repo.get('starsCount', 0)
                
                # DBに保存するタプル (name, language, stars) を作成
                all_repo_data.append((repo_name, language_name, stars_count))
                
                print(f"  [P:{current_page}/{total_pages} - R:{repo_counter}件目] {repo_name} のデータを取得...")

            current_page += 1
            
            if not repo_list and current_page > 1:
                break
        
        # --- データの挿入処理 ---
        if all_repo_data:
            cur = conn.cursor()
            # 複数データを一括挿入するために executemany を使用します。
            # IGNORE を使うことで、主キー(name)が重複してもエラーにならずスキップされます。
            sql_insert = "INSERT OR IGNORE INTO repositories (name, primary_language, stars_count) VALUES (?, ?, ?);"
            
            cur.executemany(sql_insert, all_repo_data)
            conn.commit()
            print("==================================================")
            print(f"🎉 データベースへのデータ挿入が完了しました。総件数: {len(all_repo_data)}件を処理しました。")
        else:
            print("==================================================")
            print("⚠️ 取得・保存するリポジトリデータがありませんでした。")

    except sqlite3.Error as e:
        # DB関連の例外を捕捉
        print(f"❌ データベースへのデータ挿入中にエラーが発生しました: {e}")
    
    finally:
        if conn:
            # エラーの有無に関わらず、最後にデータベース接続を閉じる
            conn.close()
            print("👋 データベース接続を閉じました。")


# --- 実行部分 ---
BASE_URL = "https://github.com/orgs/google/repositories?q=&type=all&language=&sort=stargazers"
scrape_and_save_all_google_repositories(BASE_URL)

✅ データベース `github_repos.db` に接続し、テーブルの準備が完了しました。
  [P:1/94 - R:1件目] material-design-icons のデータを取得...
  [P:1/94 - R:2件目] guava のデータを取得...
  [P:1/94 - R:3件目] zx のデータを取得...
  [P:1/94 - R:4件目] styleguide のデータを取得...
  [P:1/94 - R:5件目] leveldb のデータを取得...
  [P:1/94 - R:6件目] googletest のデータを取得...
  [P:1/94 - R:7件目] comprehensive-rust のデータを取得...
  [P:1/94 - R:8件目] material-design-lite のデータを取得...
  [P:1/94 - R:9件目] python-fire のデータを取得...
  [P:1/94 - R:10件目] flatbuffers のデータを取得...
  [P:1/94 - R:11件目] gson のデータを取得...
  [P:1/94 - R:12件目] ExoPlayer のデータを取得...
  [P:1/94 - R:13件目] iosched のデータを取得...
  [P:1/94 - R:14件目] eng-practices のデータを取得...
  [P:1/94 - R:15件目] fonts のデータを取得...
  [P:1/94 - R:16件目] filament のデータを取得...
  [P:1/94 - R:17件目] cadvisor のデータを取得...
  [P:1/94 - R:18件目] web-starter-kit のデータを取得...
  [P:1/94 - R:19件目] flexbox-layout のデータを取得...
  [P:1/94 - R:20件目] dagger のデータを取得...
  [P:1/94 - R:21件目] libphonenumber のデータを取得...
  [P:1/94 - R:22件目] gvisor のデータを取得...
  [P:1/94 - R:23件目] langextract の

### データ確認

In [2]:
import sqlite3

DB_NAME = 'github_repos.db'

def display_all_repositories():
    """
    データベースに接続し、SELECT文で全リポジトリデータを取得して表示します。
    """
    conn = None # 接続オブジェクトを初期化
    
    try:
        # データベースに接続
        conn = sqlite3.connect(DB_NAME)
        cur = conn.cursor()
        print(f"✅ データベース `{DB_NAME}` に接続しました。")

        # SQL（全レコード選択）
        sql_select = "SELECT name, primary_language, stars_count FROM repositories ORDER BY stars_count DESC;"

        # SQL文の実行
        cur.execute(sql_select)
        
    except sqlite3.Error as e:
        print('❌ データベース操作中にエラーが発生しました:', e)

    else:
        # tryブロック内でエラーが発生しなかった場合に実行される (SELECT成功時)
        print("--- 取得結果 ---")
        
        repo_count = 0
        
        # 取得した結果を一行ずつループして表示
        for row in cur:
            # rowはタプル (name, primary_language, stars_count)
            name, language, stars = row
            
            # スター数の表示形式を整える (例: 15415 -> 15.4k)
            if stars >= 1000:
                display_stars = f"{round(stars / 1000, 1):.1f}k"
            else:
                display_stars = str(stars)
                
            print(f"▶️ {name}")
            print(f"  - 主要言語: {language}")
            print(f"  - スター数: {display_stars}")
            print("-----------------------")
            
            repo_count += 1
            
        print(f"🎉 データ表示が完了しました。表示件数: {repo_count}件")

    finally:
        if conn:
            # エラーの発生の有無に関わらず実行される（接続を確実に閉じる）
            conn.close()
            print("👋 データベース接続を閉じました。")

# --- 実行部分 ---
display_all_repositories()

✅ データベース `github_repos.db` に接続しました。
--- 取得結果 ---
▶️ material-design-icons
  - 主要言語: N/A
  - スター数: 52.5k
-----------------------
▶️ guava
  - 主要言語: Java
  - スター数: 51.3k
-----------------------
▶️ zx
  - 主要言語: JavaScript
  - スター数: 44.9k
-----------------------
▶️ styleguide
  - 主要言語: HTML
  - スター数: 38.7k
-----------------------
▶️ leveldb
  - 主要言語: C++
  - スター数: 38.4k
-----------------------
▶️ googletest
  - 主要言語: C++
  - スター数: 37.5k
-----------------------
▶️ comprehensive-rust
  - 主要言語: Rust
  - スター数: 32.3k
-----------------------
▶️ material-design-lite
  - 主要言語: HTML
  - スター数: 32.2k
-----------------------
▶️ python-fire
  - 主要言語: Python
  - スター数: 28.0k
-----------------------
▶️ flatbuffers
  - 主要言語: C++
  - スター数: 25.1k
-----------------------
▶️ gson
  - 主要言語: Java
  - スター数: 24.2k
-----------------------
▶️ ExoPlayer
  - 主要言語: Java
  - スター数: 21.9k
-----------------------
▶️ iosched
  - 主要言語: Kotlin
  - スター数: 21.8k
-----------------------
▶️ eng-practices
  - 主要言語: N/A
  - スター数: 20